# PS- S6E5

- Data: https://www.kaggle.com/competitions/playground-series-s6e3/data

In [1]:
import numpy as np
import pandas as pd
import logging

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from xgboost import XGBClassifier
from catboost import CatBoostClassifier

In [2]:
# path
train_path = r"C:\Users\Rudra\Desktop\kaggle\2026-mar\p6-s6e3\data\train.csv"
test_path = r"C:\Users\Rudra\Desktop\kaggle\2026-mar\p6-s6e3\data\test.csv"
sub_path = r"C:\Users\Rudra\Desktop\kaggle\2026-mar\p6-s6e3\data\sample_submission.csv"

# load data
train = pd.read_csv(train_path)
test = pd.read_csv(test_path)
sub = pd.read_csv(sub_path)

# Columns selection
object_cols = train.select_dtypes(include="object").columns.to_list()
object_cols.remove('Churn')

num_cols = train.select_dtypes(exclude="object").columns.to_list()
num_cols.remove('id')

target_col = 'Churn'


In [3]:
logging.basicConfig(
    filename="training.log",
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)

logging.info("Training started")

In [4]:
train = pd.read_csv(train_path)
test = pd.read_csv(test_path)
sub = pd.read_csv(sub_path)

target_col = "Churn"

X = train.drop(columns=[target_col])
y = train[target_col].map({'No':0, 'Yes':1})

In [5]:
def train_model_cv(model, X, y, X_test, n_splits=5):
    
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    
    oof_preds = np.zeros(len(X))
    test_preds = np.zeros(len(X_test))
    fold_scores = []
    
    for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
        
        logging.info(f"Starting Fold {fold+1}")
        
        X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
        
        model.fit(X_train, y_train)
        
        val_pred = model.predict_proba(X_val)[:, 1]
        oof_preds[val_idx] = val_pred
        
        fold_auc = roc_auc_score(y_val, val_pred)
        fold_scores.append(fold_auc)
        
        logging.info(f"Fold {fold+1} AUC: {fold_auc:.5f}")
        
        test_preds += model.predict_proba(X_test)[:, 1] / n_splits
    
    overall_auc = roc_auc_score(y, oof_preds)
    
    logging.info(f"Overall CV AUC: {overall_auc:.5f}")
    
    return oof_preds, test_preds, fold_scores, overall_auc

In [6]:
preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(drop="first", handle_unknown="ignore"), object_cols),
        ("num", StandardScaler(), num_cols),
    ]
)

In [7]:
rf_model = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("model", RandomForestClassifier(
        n_estimators=300,
        max_depth=None,
        random_state=42,
        n_jobs=-1
    ))
])

oof_rf, test_rf, fold_scores_rf, cv_rf = train_model_cv(
    rf_model, X, y, test
)

results = []

results.append({
    "model": "RandomForest",
    "cv_auc": cv_rf,
    "fold_std": np.std(fold_scores_rf)
})

print("RF CV:", cv_rf)

RF CV: 0.8970330028982391


In [8]:
xgb_model = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("model", XGBClassifier(
        n_estimators=500,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric="logloss",
        random_state=42,
        n_jobs=-1
    ))
])

oof_xgb, test_xgb, fold_scores_xgb, cv_xgb = train_model_cv(
    xgb_model, X, y, test
)

results.append({
    "model": "XGBoost",
    "cv_auc": cv_xgb,
    "fold_std": np.std(fold_scores_xgb)
})

print("XGB CV:", cv_xgb)

XGB CV: 0.9163509348253641


In [9]:
cat_features_idx = [X.columns.get_loc(col) for col in object_cols]

In [10]:
def train_catboost_cv(X, y, X_test, n_splits=5):
    
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    
    oof_preds = np.zeros(len(X))
    test_preds = np.zeros(len(X_test))
    fold_scores = []
    
    for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
        
        X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
        
        model = CatBoostClassifier(
            iterations=500,
            learning_rate=0.05,
            depth=6,
            eval_metric="AUC",
            verbose=0,
            random_state=42
        )
        
        model.fit(
            X_train, y_train,
            cat_features=cat_features_idx
        )
        
        val_pred = model.predict_proba(X_val)[:,1]
        oof_preds[val_idx] = val_pred
        
        fold_auc = roc_auc_score(y_val, val_pred)
        fold_scores.append(fold_auc)
        
        test_preds += model.predict_proba(X_test)[:,1] / n_splits
    
    overall_auc = roc_auc_score(y, oof_preds)
    
    return oof_preds, test_preds, fold_scores, overall_auc

In [11]:
oof_cat, test_cat, fold_scores_cat, cv_cat = train_catboost_cv(
    X, y, test
)

results.append({
    "model": "CatBoost",
    "cv_auc": cv_cat,
    "fold_std": np.std(fold_scores_cat)
})

print("CatBoost CV:", cv_cat)

CatBoost CV: 0.9154045480056804


In [12]:
results_df = pd.DataFrame(results)
results_df = results_df.sort_values(by="cv_auc", ascending=False)

print(results_df)

          model    cv_auc  fold_std
1       XGBoost  0.916351  0.000967
2      CatBoost  0.915405  0.000968
0  RandomForest  0.897033  0.000948


In [14]:
ensemble_oof = (
    0.4 * oof_xgb +
    0.5 * oof_cat +
    0.1 * oof_rf
)

ensemble_auc = roc_auc_score(y, ensemble_oof)
print("Ensemble CV AUC:", ensemble_auc)

Ensemble CV AUC: 0.9159705023486533


In [16]:
final_test_pred = (
    0.9 * test_xgb +
    0.1 * test_cat
)

sub["Churn"] = final_test_pred
sub.to_csv("submission_ensemble_2.csv", index=False)

print("Ensemble submission saved.")

Ensemble submission saved.


In [15]:
best_auc = 0
best_weights = None

for w1 in np.arange(0, 1.1, 0.1):
    for w2 in np.arange(0, 1.1 - w1, 0.1):
        w3 = 1 - w1 - w2
        
        ensemble = (
            w1 * oof_xgb +
            w2 * oof_cat +
            w3 * oof_rf
        )
        
        auc = roc_auc_score(y, ensemble)
        
        if auc > best_auc:
            best_auc = auc
            best_weights = (w1, w2, w3)

print("Best AUC:", best_auc)
print("Best Weights:", best_weights)

Best AUC: 0.9163720533904212
Best Weights: (0.9, 0.1, -2.7755575615628914e-17)
